# Default notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
# Read sample data
df = spark.table("samples.bakehouse.sales_customers")
display(df.limit(5))

In [0]:
from pyspark.sql import functions as F

# Business logic: Group by country, count customers by gender
customers = df \
  .groupBy("country") \
  .agg(
    F.count("*").alias("total_customers"),
    F.count_if(F.col("gender") == "female").alias("female_customers"),
    F.count_if(F.col("gender") == "male").alias("male_customers")
  )

# Write to Unity Catalog
customers.write.mode("overwrite") \
  .saveAsTable(f"{catalog}.{schema}.customer_360")

In [0]:
display(spark.table(f"{catalog}.{schema}.customer_360"))